# CASMI 2026 · a pipeline found by an autonomous research loop (pi-rsi)

This notebook is the best pipeline found so far by **[pi-rsi](https://github.com/xiehuanyi/pi-rsi)**, an open-source
automated research harness: a research-lead model proposes one hypothesis at a time, a coding agent implements it in
its own git branch, a frozen evaluator scores it, and the loop decides where to go next. No human wrote or tuned the
model code below; humans built the local benchmark and the harness. Everything ran on one workstation
(RTX 3090 + RTX 2080 Ti).

**Public LB: 0.228** (2026-10-14, rank 2341 / 3307; top 0.495). The local proxy score of 0.400 did **not** transfer:
our closed-world benchmark restricts candidates to training structures ∪ COCONUT (no PubChem), balances the three
novelty classes, and uses mostly public Q-TOF queries, so it rewards different things than the hidden test. We publish
the pipeline as an honest data point about autonomous research on a proxy benchmark, not as a strong solution.

## Local benchmark (closed world, frozen before the search)
Held-out natural products from `enveda-np-examples` (timsTOF) and COCONUT molecules with public Q-TOF spectra,
split into dev / validation / sealed final, with the competition's three novelty classes in equal shares:
1 = other-instrument library spectra remain in training, 2 = all spectra removed but the structure is in the
candidate DB (COCONUT), 3 = removed from everything. Metric = the competition's MRR@25 (tautomer-canonical
InChIKey14). Local scores are not comparable to the LB (class mix and candidate DB differ).

| node | change | val MRR@25 |
|---|---|---:|
| root | mass window + binned cosine library search, NP-likeness prior | 0.307 |
| n001 | + neural spectrum → Morgan-fingerprint ranker blended with library matching | 0.352 |
| n002 | + dual-encoder contrastive ranking with formula/mass hard negatives | 0.378 |
| n003 | + disagreement veto for weak library promotions | 0.383 |
| n005 | + complementary exact-mass peak ranker | 0.396 |
| n007 | + fragment-only verification of strong library promotions | **0.400** (class 1/2/3: 0.836 / 0.369 / 0) |

## What the pipeline does
1. Neutral mass from precursor m/z and adduct → candidates within 10 ppm from training structures ∪ COCONUT.
2. Library matching: best binned cosine against each candidate's training spectra; strong matches are promoted and
   then verified on fragment peaks only.
3. A fingerprint MLP (two-member ensemble) predicts a 2048-bit Morgan fingerprint from each spectrum; a contrastive
   dual encoder is fine-tuned on same-mass hard negatives; an exact-mass peak encoder adds a third view.
4. The views are blended per molecule and the top 25 unique structures (by metric key) are submitted.
The pipeline writes a valid library-search submission first and replaces it only when later stages succeed.

## Inputs
- Competition data; RDKit 2026.03.3 wheel (the metric's own dataset).
- COCONUT 2.0 (Oct 2026, CC0) with precomputed metric keys — please cite Chandrasekhar et al., *COCONUT 2.0*,
  NAR 2024, https://doi.org/10.1093/nar/gkae1063.

Runtime ≈ 4.3 h on 2×T4 (data keying ~32 min; the fingerprint MLP is I/O-bound on Kaggle's disk). No internet, no pretrained models.

In [ ]:
import glob, os, subprocess, sys, time
T0 = time.time()
COMP = glob.glob('/kaggle/input/**/enveda-CASMI26-molecule-id-mass-spectra', recursive=True)[0]
COCONUT = glob.glob('/kaggle/input/**/coconut_key14.parquet', recursive=True)[0]
WHEEL = glob.glob('/kaggle/input/**/rdkit-2026.3.3-cp%d%d-*.whl' % sys.version_info[:2], recursive=True)[0]
WORK = '/tmp/casmi'                      # large scratch (training corpus memmap ~25 GB); /kaggle/working is 20 GB
os.makedirs('/kaggle/working/pipeline/src', exist_ok=True)
os.chdir('/kaggle/working/pipeline')
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', WHEEL],
                     capture_output=True, text=True).stdout[-300:])
import rdkit; print('rdkit', rdkit.__version__, '| inputs:', COMP, COCONUT)

## Pipeline code
The files below are the committed code of the pi-rsi node, unchanged.

In [ ]:
%%writefile kaggle_prepare.py
#!/usr/bin/env python3
"""Build the pipeline's data directory on Kaggle (4 CPUs, ~31 GB RAM) from the competition train.parquet and the
COCONUT key table. Same layout as the local benchmark's data/worker, but with ALL training spectra:

  train_spectra.parquet  every training row + key14 (metric key), sorted by key14, zstd row groups of 50k
  candidates.parquet     training structures ∪ COCONUT, deduplicated by key14, sorted by neutral monoisotopic mass

key14 = first InChIKey block after RDKit 2026.03.3 TautomerEnumerator canonicalization (the CASMI metric key).
Sorting is done in key-range buckets so peak memory stays a fraction of the table.
"""
from __future__ import annotations

import argparse
import os
import time
from multiprocessing import get_context
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

BUCKETS = 8


def _init():
    global _Chem, _Desc, _canon
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdMolDescriptors
    from rdkit.Chem.MolStandardize import rdMolStandardize
    RDLogger.DisableLog('rdApp.*')
    _Chem, _Desc, _canon = Chem, rdMolDescriptors, rdMolStandardize.TautomerEnumerator()


def _annotate(smiles):
    try:
        mol = _Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, None, None
        key = _Chem.MolToInchiKey(_canon.Canonicalize(mol))[:14] or None
        return key, _Desc.CalcMolFormula(mol), float(_Desc.CalcExactMolWt(mol))
    except Exception:
        return None, None, None


def annotate(smiles, procs):
    with get_context('fork').Pool(procs, initializer=_init) as pool:
        out = pool.map(_annotate, smiles, chunksize=256)
    return pd.DataFrame(out, columns=['key14', 'formula', 'mono_mass'])


def write_sorted(train: Path, key_of: dict, out: Path):
    """Add key14 and write sorted by key14, one key-range bucket at a time."""
    pf = pq.ParquetFile(train)
    keys = sorted({k for k in key_of.values() if k})
    edges = [keys[int(len(keys) * i / BUCKETS)] for i in range(1, BUCKETS)]
    schema = pf.schema_arrow.append(pa.field('key14', pa.string()))
    with pq.ParquetWriter(out, schema, compression='zstd') as writer:
        for b in range(BUCKETS):
            lo, hi = (edges[b - 1] if b else None), (edges[b] if b < BUCKETS - 1 else None)
            parts = []
            for rg in range(pf.num_row_groups):
                t = pf.read_row_group(rg)
                k = pa.array([key_of.get(s) for s in t.column('normalized_smiles').to_pylist()], pa.string())
                mask = pc.is_valid(k) if lo is None else pc.greater_equal(k, lo)
                if hi is not None:
                    mask = pc.and_(mask, pc.less(k, hi))
                mask = pc.fill_null(mask, False)
                parts.append(t.append_column('key14', k).filter(mask))
            part = pa.concat_tables(parts).sort_by([('key14', 'ascending')])
            writer.write_table(part, row_group_size=50_000)
            print(f'[prepare] bucket {b + 1}/{BUCKETS}: {part.num_rows} rows', flush=True)
            del parts, part


def candidates(st: pd.DataFrame, coconut: Path) -> pd.DataFrame:
    tr = st.dropna(subset=['key14', 'mono_mass']).drop_duplicates('key14')
    tr = tr.rename(columns={'normalized_smiles': 'smiles'})[['key14', 'smiles', 'formula', 'mono_mass']]
    co = pd.read_parquet(coconut, columns=['key14', 'smiles', 'formula', 'mono_mass', 'coconut_id', 'np_likeness',
                                           'np_pathway']).dropna(subset=['key14', 'mono_mass']).drop_duplicates('key14')
    c = tr.merge(co, on='key14', how='outer', suffixes=('', '_coco'))
    for col in ('smiles', 'formula', 'mono_mass'):
        c[col] = c[col].fillna(c.pop(col + '_coco'))
    c['in_train'] = c['key14'].isin(set(tr['key14']))
    c['in_coconut'] = c['coconut_id'].notna()
    return c.sort_values('mono_mass').reset_index(drop=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--train', required=True)
    ap.add_argument('--coconut', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--procs', type=int, default=os.cpu_count() or 4)
    a = ap.parse_args()
    t0, out = time.time(), Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    smiles = pq.read_table(a.train, columns=['normalized_smiles']).column('normalized_smiles')
    uniq = pc.unique(smiles.combine_chunks()).drop_null().to_pylist()
    del smiles
    print(f'[prepare] {len(uniq)} unique training structures; keying with {a.procs} processes', flush=True)
    st = annotate(uniq, a.procs)
    st.insert(0, 'normalized_smiles', uniq)
    print(f'[prepare] keyed in {time.time() - t0:.0f}s, failed {int(st.key14.isna().sum())}', flush=True)
    write_sorted(Path(a.train), dict(zip(st['normalized_smiles'], st['key14'])), out / 'train_spectra.parquet')
    cands = candidates(st, Path(a.coconut))
    cands.to_parquet(out / 'candidates.parquet', index=False)
    print(f'[prepare] candidates {len(cands)} (train {int(cands.in_train.sum())}, coconut {int(cands.in_coconut.sum())}); '
          f'done in {time.time() - t0:.0f}s', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile main.py
#!/usr/bin/env python3
"""Protected fingerprint/library ranker plus fresh mass-hard-negative dual-encoder ranking."""
from __future__ import annotations

import argparse
import hashlib
import json
import time
import traceback
from pathlib import Path

import numpy as np

import pandas as pd

from src.masses import consensus_mass, neutral_mass
from src.rank import CandidateIndex, load_library, rank_molecule
from src.spectra import binned, clean
from src.blend import blend_order, fingerprint_scores, library_scores
from src.absolute_gate import AbsolutePolicy, DEFAULT_ABSOLUTE
from src.fragment import DEFAULT_FRAGMENT

FALLBACK_SMILES = 'C'  # every molecule needs at least one guess


def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument('--data-dir', required=True)
    ap.add_argument('--queries', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--work-dir', required=True)
    ap.add_argument('--budget', type=float, required=True, help='wall-clock seconds for this whole process')
    ap.add_argument('--seed', type=int, default=0)
    return ap.parse_args()


def main():
    a = parse_args()
    t0 = time.monotonic()
    np.random.seed(a.seed)
    import pyarrow as pa
    pa.set_cpu_count(16)
    pa.set_io_thread_count(8)
    work = Path(a.work_dir)
    work.mkdir(parents=True, exist_ok=True)
    Path(a.out).parent.mkdir(parents=True, exist_ok=True)
    deadline = t0 + a.budget * 0.89
    data = Path(a.data_dir)
    q = pd.read_parquet(a.queries).reset_index(drop=True)
    q['neutral_mass'] = [neutral_mass(m, ad) for m, ad in zip(q['precursor_mz'], q['adduct'])]
    mols = q.groupby('molecule_id', sort=False)
    cands = pd.read_parquet(data / 'candidates.parquet', columns=['key14', 'smiles', 'formula', 'mono_mass', 'np_likeness', 'in_train'])
    index = CandidateIndex(cands.dropna(subset=['key14', 'mono_mass']))
    windows = {mid: index.window(consensus_mass(g['neutral_mass'])) for mid, g in mols}
    keys = set().union(*(set(w['key14']) for w in windows.values()))
    diagnostic_library = {} if DEFAULT_FRAGMENT.enabled and a.budget >= 1500 else None
    lib = load_library(data / 'train_spectra.parquet', keys, diagnostic_library)
    print(f'[baseline] {len(windows)} molecules, {len(keys)} window candidates, '
          f'{len(lib)} with library spectra, {time.monotonic() - t0:.0f}s', flush=True)
    rows = []
    query_library = {}
    for mid, g in mols:
        qs = []
        for r in g.itertuples():
            mz, w = clean(r.ms2_mzs, r.ms2_normalized_intensities, r.precursor_mz)
            if w.size:
                qs.append((r.ionization_mode, binned(mz, w)))
        query_library[mid] = qs
        guesses = rank_molecule(qs, windows[mid], lib) or [FALLBACK_SMILES]
        rows.append({'molecule_id': mid, 'smiles': ';'.join(guesses)})
    # Valid root output is available before any expensive processing or GPU failure.
    pd.DataFrame(rows).to_csv(a.out, index=False)
    if a.budget < 300:
        return
    try:
        from src.fingerprints import build_fingerprints
        from src.features import query_features
        from src.learner import prepare_training, train_model, predict
        targets = cands[cands['in_train'] | cands['key14'].isin(keys)].reset_index(drop=True)
        fps = build_fingerprints(targets['smiles'].tolist())
        fp_index = {k: i for i, k in enumerate(targets['key14'])}
        train_targets = targets[targets['in_train']]
        train_fps = fps[train_targets.index]
        full_scale = a.budget >= 1500
        x, ids, weights, holdout, public = prepare_training(
            data / 'train_spectra.parquet', train_targets['key14'].tolist(), work, a.seed,
            min(deadline - 45, t0 + a.budget * 0.55), max_per_key=0 if full_scale else 6)
        print(f'[learn] preprocessing done in {time.monotonic() - t0:.1f}s', flush=True)
        query_x = query_features(q)
        predictions = []
        base_started = time.monotonic()
        contrast_base = None
        source_weights = (2.0, 4.0) if full_scale else (2.0,)
        for member, source_weight in enumerate(source_weights):
            # Second member stresses public/NP training, but inference uses no source labels.
            member_weights = weights * np.where(public, source_weight / 2.0, 1.0)
            remaining_members = len(source_weights) - member
            base_end = min(deadline - 20, t0 + a.budget * 0.58)
            train_deadline = time.monotonic() + (base_end - time.monotonic()) / remaining_members
            print(f'[learn] member={member}, public_weight={source_weight}', flush=True)
            try:
                model, log_weight = train_model(x, ids, member_weights, holdout, train_fps,
                                                a.seed + member * 31, train_deadline)
                predictions.append(predict(model, log_weight, query_x))
                if member == 0:
                    contrast_base = model.cpu()
                    contrast_log_weight = log_weight.cpu()
                del model, log_weight
            except Exception:
                if not predictions:
                    raise
                traceback.print_exc()
                print('[learn] retaining completed ensemble members', flush=True)
                break
        predictions = np.mean(predictions, axis=0)
        print(f'[timing] BCE ensemble incl query inference={time.monotonic() - base_started:.1f}s', flush=True)
    except Exception:
        traceback.print_exc()
        print('[learn] fallback to root output', flush=True)
        return
    try:
        rows = neural_rows(mols, windows, fp_index, fps, predictions, query_library, lib)
        temporary = work / 'submission_neural.csv'
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(a.out)
    except Exception:
        traceback.print_exc()
        print('[blend] keeping root output after reranking/output failure', flush=True)
        return
    print(f'[blend] protected n001 output in {time.monotonic() - t0:.0f}s', flush=True)
    contrast_complete = False
    try:
        import torch
        from src.contrastive import train_contrastive, predict_embeddings, BLEND_WEIGHT
        if contrast_base is None or time.monotonic() > deadline - 60:
            return
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        contrast_base = contrast_base.to(device)
        contrast_log_weight = contrast_log_weight.to(device)
        contrast_started = time.monotonic()
        dual = train_contrastive(contrast_base, contrast_log_weight, x, ids, weights, holdout,
                                train_fps, train_targets['mono_mass'].to_numpy(),
                                train_targets['formula'].fillna('').to_numpy(), a.seed,
                                deadline - 20)
        inference_started = time.monotonic()
        print(f'[timing] contrastive incl pools/validation={inference_started - contrast_started:.1f}s', flush=True)
        spec_embed, struct_embed = predict_embeddings(dual, query_x, fps)
        rerank_started = time.monotonic()
        print(f'[timing] embedding inference={rerank_started - inference_started:.1f}s', flush=True)
        rows = neural_rows(mols, windows, fp_index, fps, predictions, query_library, lib,
                           spec_embed, struct_embed, BLEND_WEIGHT)
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(a.out)
        parent = work / 'submission_parent.csv'
        pd.DataFrame(rows).to_csv(parent, index=False)
        # Unlabelled exact rows survive evaluator scratch cleanup for paired controls.
        print('[parent] ' + json.dumps(rows), flush=True)
        print('[parent_sha256] ' + hashlib.sha256(parent.read_bytes()).hexdigest(), flush=True)
        contrast_complete = True
        print(f'[timing] contrastive rerank/write={time.monotonic() - rerank_started:.1f}s', flush=True)
    except Exception:
        traceback.print_exc()
        print('[contrast] retaining protected n001 output', flush=True)
    print(f'[blend] protected n003 wrote {len(rows)} rows in {time.monotonic() - t0:.0f}s', flush=True)
    # Optional work starts only AFTER a complete protected n003 output exists.
    if contrast_complete and a.budget >= 1500 and time.monotonic() < deadline - 300:
        try:
            from src.peak_features import prepare_peaks, query_peaks
            from src.peak_train import train_peak, predict_peak
            from src.peak_config import BLEND_WEIGHT as PEAK_WEIGHT
            peak_started = time.monotonic()
            peaks = prepare_peaks(data / 'train_spectra.parquet', work, deadline - 240)
            peak_model = train_peak(peaks, x[:, -24:], ids, weights, holdout, train_fps,
                                    train_targets['mono_mass'].to_numpy(),
                                    train_targets['formula'].fillna('').to_numpy(), a.seed,
                                    deadline - 40)
            peak_spec, peak_struct = predict_peak(peak_model, query_peaks(q), query_x[:, -24:],
                                                 fps, deadline - 15)
            rows = neural_rows(mols, windows, fp_index, fps, predictions, query_library, lib,
                               spec_embed, struct_embed, BLEND_WEIGHT,
                               peak_spec, peak_struct, PEAK_WEIGHT,
                               absolute_policy=AbsolutePolicy())
            if time.monotonic() > deadline - 5:
                raise TimeoutError('optional peak write deadline')
            pd.DataFrame(rows).to_csv(temporary, index=False)
            if time.monotonic() > deadline:
                raise TimeoutError('optional peak atomic replacement deadline')
            temporary.replace(a.out)
            # Exact n005 exists before any new policy work; its rows and digest
            # survive scratch cleanup for the preregistered paired control.
            print('[n005] ' + json.dumps(rows), flush=True)
            print('[n005_sha256] ' + hashlib.sha256(Path(a.out).read_bytes()).hexdigest(), flush=True)
            if DEFAULT_ABSOLUTE.enabled and time.monotonic() < deadline - 15:
                try:
                    policy_records = {}
                    policy_rows = neural_rows(
                        mols, windows, fp_index, fps, predictions, query_library, lib,
                        spec_embed, struct_embed, BLEND_WEIGHT, peak_spec, peak_struct, PEAK_WEIGHT,
                        absolute_policy=DEFAULT_ABSOLUTE, policy_deadline=deadline - 5,
                        policy_records=policy_records)
                    pd.DataFrame(policy_rows).to_csv(temporary, index=False)
                    if time.monotonic() > deadline - 5:
                        raise TimeoutError('absolute policy replacement deadline')
                    temporary.replace(a.out)
                    print('[n006] ' + json.dumps(policy_rows), flush=True)
                    print('[n006_sha256] ' + hashlib.sha256(Path(a.out).read_bytes()).hexdigest(), flush=True)
                    try:
                        from src.fragment_pass import optional_fragment_pass
                        optional_fragment_pass(Path(a.out), work, deadline, q, windows,
                                               diagnostic_library, policy_records)
                    except Exception:
                        traceback.print_exc()
                        print('[fragment] retaining complete n006 output after failure', flush=True)
                except Exception:
                    traceback.print_exc()
                    print('[absolute] retaining complete n005 output', flush=True)
            print(f'[peak] optional total={time.monotonic() - peak_started:.1f}s; '
                  f'process={time.monotonic() - t0:.1f}s', flush=True)
        except Exception:
            traceback.print_exc()
            print('[peak] retaining protected n003 output', flush=True)


def neural_rows(mols, windows, fp_index, fps, predictions, query_library, lib,
                spec_embed=None, struct_embed=None, contrast_weight=0.0,
                peak_spec=None, peak_struct=None, peak_weight=0.0,
                absolute_policy=DEFAULT_ABSOLUTE, policy_deadline=None, policy_records=None):
    """Build the complete learned submission before atomically replacing the valid fallback."""
    rows = []
    for mid, g in mols:
        if policy_deadline is not None and time.monotonic() > policy_deadline:
            raise TimeoutError('absolute policy rerank deadline')
        hits = windows[mid]
        ix = [fp_index[k] for k in hits['key14']]
        # Equal spectrum weighting; no novelty/source metadata reaches inference.
        prediction = predictions[g.index].mean(axis=0)
        fs = fingerprint_scores(prediction, fps[ix].astype(np.float32))
        ls = library_scores(query_library[mid], hits, lib)
        cs = None if spec_embed is None else struct_embed[ix] @ spec_embed[g.index].mean(axis=0)
        spectrum_fp = None if cs is None else np.asarray([
            fingerprint_scores(p, fps[ix].astype(np.float32)) for p in predictions[g.index]])
        spectrum_contrast = None if cs is None else spec_embed[g.index] @ struct_embed[ix].T
        ps = None if peak_spec is None else peak_struct[ix] @ peak_spec[g.index].mean(axis=0)
        spectrum_peak = None if ps is None else peak_spec[g.index] @ peak_struct[ix].T
        gate, post_peak_order = {}, []
        order = blend_order(fs, ls, hits['np_likeness'], cs, contrast_weight, diagnostics=gate,
                            spectrum_fp=spectrum_fp, spectrum_contrast=spectrum_contrast,
                            peak=ps, peak_weight=peak_weight, spectrum_peak=spectrum_peak,
                            absolute_policy=absolute_policy, post_peak_order=post_peak_order)
        guesses = hits.iloc[order]['smiles'].tolist() or [FALLBACK_SMILES]
        rows.append({'molecule_id': mid, 'smiles': ';'.join(guesses)})
        # Unlabelled diagnostics in the run log permit paired development error analysis.
        record = {'id': mid, 'keys': hits['key14'].tolist(), 'order': order.tolist(),
              'fp': fs.tolist(), 'lib': ls.tolist(),
              'contrast': None if cs is None else cs.tolist(), 'gate': gate,
              'peak': None if ps is None else ps.tolist(), 'peak_weight': peak_weight,
              'post_peak_order': post_peak_order,
              'spectrum_fp': None if spectrum_fp is None else spectrum_fp.tolist(),
              'spectrum_contrast': None if spectrum_contrast is None else spectrum_contrast.tolist(),
              'spectrum_peak': None if spectrum_peak is None else spectrum_peak.tolist(),
              'prior': hits['np_likeness'].to_numpy(dtype=float).tolist()}
        if policy_records is not None:
            policy_records[mid] = record
        print('[rank] ' + json.dumps(record), flush=True)
    return rows


if __name__ == '__main__':
    main()

In [ ]:
open('src/__init__.py', 'w').close()  # empty module

In [ ]:
%%writefile src/absolute_gate.py
"""Conservative three-view release of absolute library promotions (inference only)."""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np


@dataclass(frozen=True)
class AbsolutePolicy:
    enabled: bool = False
    margin: float = 0.5


DEFAULT_ABSOLUTE = AbsolutePolicy(True, 1.0)
COSINE_CEILING = 0.90
RANK_CUTOFF = 3
SD_FLOOR = 0.025


def view_support(scores, library_top, challenger, margin):
    """Strict competition rank and a positive standardized challenger gap."""
    try:
        x = np.asarray(scores, dtype=float)
    except (TypeError, ValueError):
        return None
    if x.ndim != 1 or not len(x) or not np.isfinite(x).all():
        return None
    if not (0 <= library_top < len(x) and 0 <= challenger < len(x)):
        return None
    rank = int(1 + (x > x[library_top]).sum())
    gap = float((x[challenger] - x[library_top]) / max(x.std(), SD_FLOOR))
    return {'rank': rank, 'margin': gap,
            'supports': bool(rank > RANK_CUTOFF and gap > 0 and gap >= margin)}


def absolute_disagreement(fp, contrast, peak, library_top, challenger, cosine, policy,
                          spectrum_fp=None, spectrum_contrast=None, spectrum_peak=None):
    """All aggregate views and a strict majority of identical-spectrum votes must agree.

    Missing/malformed scores retain the promotion. The views are correlated,
    so their agreement is a conservative policy, not independent evidence.
    """
    info = {'released': False, 'challenger': int(challenger), 'votes': 0,
            'n_spectra': 0, 'views': {}, 'spectrum_votes': []}
    if (not policy.enabled or not np.isfinite(cosine) or not 0.70 <= cosine < COSINE_CEILING
            or library_top == challenger):
        return info
    aggregate = (fp, contrast, peak)
    if any(x is None for x in aggregate):
        return info
    try:
        aggregate = [np.asarray(x, dtype=float) for x in aggregate]
    except (TypeError, ValueError):
        return info
    n = len(fp)
    for name, x in zip(('fp', 'contrast', 'peak'), aggregate):
        if np.shape(x) != (n,):
            return info
        result = view_support(x, library_top, challenger, policy.margin)
        info['views'][name] = result
        if result is None:
            return info
    spectra = (spectrum_fp, spectrum_contrast, spectrum_peak)
    if any(x is None for x in spectra):
        return info
    try:
        arrays = [np.asarray(x, dtype=float) for x in spectra]
    except (TypeError, ValueError):
        return info
    shape = arrays[0].shape
    if len(shape) != 2 or shape[0] < 1 or shape[1] != n:
        return info
    if any(x.shape != shape or not np.isfinite(x).all() for x in arrays):
        return info
    votes = []
    for scores in zip(*arrays):
        results = [view_support(x, library_top, challenger, policy.margin) for x in scores]
        votes.append(all(r is not None and r['supports'] for r in results))
    info.update(votes=int(sum(votes)), n_spectra=len(votes), spectrum_votes=votes)
    info['released'] = bool(all(r['supports'] for r in info['views'].values())
                            and info['votes'] * 2 > len(votes))
    return info

In [ ]:
%%writefile src/blend.py
"""Fingerprint agreement plus root cosine, with an explicit confident-library gate."""
from __future__ import annotations

import numpy as np

from .spectra import cosine
from .library_gate import DEFAULT_VETO, disagreement
from .absolute_gate import DEFAULT_ABSOLUTE, absolute_disagreement

LIB_WEIGHT = 1.5
GATE_COSINE = 0.70
GATE_MIN = 0.35
GATE_MARGIN = 0.10
PRIOR_WEIGHT = 0.08
LIB_RANK_WEIGHT = 2.0  # soft rescue of weak-library candidates, not library-first ordering


def library_scores(queries, hits, library):
    scores = np.full(len(hits), -1.0)
    for i, key in enumerate(hits['key14']):
        for mode, ref in library.get(key, ()):
            for qmode, q in queries:
                if mode == qmode:
                    scores[i] = max(scores[i], cosine(q, ref))
    return scores


def fingerprint_scores(prediction, fps):
    # Cosine avoids the strong short-fingerprint preference of unnormalized Bernoulli likelihood.
    return (fps @ prediction) / np.maximum(np.sqrt(fps.sum(axis=1)), 1) / max(np.linalg.norm(prediction), 1e-8)


def gate_regime(lib, library_order):
    """Original n002 gate clauses, with the absolute clause taking precedence."""
    top = library_order[0]
    runnerup = lib[library_order[1]] if len(lib) > 1 else 0.0
    if lib[top] >= GATE_COSINE:
        return 'absolute'
    if lib[top] >= GATE_MIN and lib[top] - runnerup >= GATE_MARGIN:
        return 'margin'
    if lib[top] >= 0.10 and runnerup < 0:
        return 'missing_runnerup'
    return 'none'


def blend_order(fp, lib, prior, contrast=None, contrast_weight=0.0,
                veto=DEFAULT_VETO, diagnostics=None, spectrum_fp=None, spectrum_contrast=None,
                peak=None, peak_weight=0.0, spectrum_peak=None,
                absolute_policy=DEFAULT_ABSOLUTE, post_peak_order=None):
    if not len(fp):
        return np.array([], dtype=int)
    learned = (fp - fp.mean()) / max(fp.std(), 0.025)
    raw_prior = np.asarray(prior, dtype=float)
    prior = np.nan_to_num(raw_prior, nan=0.0)
    library_order = np.lexsort((-np.nan_to_num(raw_prior, nan=-10.0), -lib))
    library_rank = np.empty(len(lib), dtype=float)
    library_rank[library_order] = np.arange(1, len(lib) + 1)
    score = learned + LIB_WEIGHT * np.maximum(lib - 0.25, 0) + PRIOR_WEIGHT * np.clip(prior, -3, 3)
    score += LIB_RANK_WEIGHT * 3 / (2 + library_rank) * (lib >= 0)
    if contrast is not None and contrast_weight:
        contrast = np.asarray(contrast, dtype=float)
        score += contrast_weight * (contrast - contrast.mean()) / max(contrast.std(), 0.025)
    order = np.argsort(-score, kind='stable')
    top = library_order[0]
    regime = gate_regime(lib, library_order)
    info = disagreement(fp, contrast if contrast_weight else None, top, order[0], veto,
                        spectrum_fp, spectrum_contrast)
    # The inherited WEAK veto never releases an absolute high-cosine match.
    info['released'] = bool(info['released'] and regime in ('margin', 'missing_runnerup'))
    if diagnostics is not None:
        diagnostics.update(info, regime=regime, library_top=int(top), blended_top=int(order[0]))
    # Gate decisions above use ONLY the original n003 blended winner and views.
    # Zero weight is an exact no-op (including stable ties and missing fallbacks).
    if peak is not None and peak_weight:
        peak = np.asarray(peak, dtype=float)
        if peak.shape != fp.shape or not np.isfinite(peak).all():
            raise ValueError('invalid optional peak candidate scores')
        score += peak_weight * (peak - peak.mean()) / max(peak.std(), 0.025)
        order = np.argsort(-score, kind='stable')
    if post_peak_order is not None:
        post_peak_order.extend(order.tolist())
    # Keep n003's weak releases unchanged. Only an otherwise retained absolute
    # promotion may be challenged by the FULL post-peak blended winner.
    absolute = None
    if regime == 'absolute' and peak is not None and peak_weight and absolute_policy.enabled:
        absolute = absolute_disagreement(
            fp, contrast if contrast_weight else None, peak, top, order[0], lib[top],
            absolute_policy, spectrum_fp, spectrum_contrast, spectrum_peak)
        if diagnostics is not None:
            diagnostics['absolute'] = absolute
    released = info['released'] or (absolute is not None and absolute['released'])
    if regime != 'none' and not released:
        order = np.r_[top, order[order != top]]
    return order[:25]

In [ ]:
%%writefile src/contrastive.py
"""From-scratch candidate-aware dual encoder, initialized from the invocation's BCE model."""
from __future__ import annotations

import copy
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

from .hard_negatives import negative_table

EMBED_DIM = 256
RANK_WEIGHT = 0.20
BLEND_WEIGHT = 1.0
HARD_COUNT = 16
EPOCH_SAMPLES = 262144
BATCH = 512
MAX_EPOCHS = 40
TEMPERATURE = 0.07


class DualEncoder(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.spectrum = copy.deepcopy(base)
        self.project = nn.Sequential(nn.Linear(1024, 512), nn.ReLU(), nn.Linear(512, EMBED_DIM))
        self.structure = nn.Sequential(nn.Linear(2048, 512), nn.ReLU(), nn.Linear(512, EMBED_DIM))

    def encode_spectrum(self, x):
        hidden = self.spectrum.network[:-1](x)
        logits = self.spectrum.network[-1](hidden)
        return F.normalize(self.project(hidden).float(), dim=-1), logits

    def encode_structure(self, fp):
        return F.normalize(self.structure(fp.float()).float(), dim=-1)


def ranking_loss(spec, positive, hard, key_ids, temperature=TEMPERATURE):
    # Ordinary in-batch negatives plus per-spectrum formula/mass alternatives.
    ordinary = spec @ positive.T / temperature
    same_key = key_ids[:, None] == key_ids[None, :]
    ordinary = ordinary.masked_fill(same_key, -1e4)
    true = (spec * positive).sum(-1, keepdim=True) / temperature
    difficult = torch.einsum('bd,bnd->bn', spec, hard) / temperature
    logits = torch.cat([true, difficult, ordinary], dim=1)
    return F.cross_entropy(logits, torch.zeros(len(spec), dtype=torch.long, device=spec.device))


def _evaluate(model, features, ids, fps, val, pools, log_weight, device):
    model.eval()
    bce = reciprocal = count = 0
    with torch.no_grad():
        for lo in range(0, len(val), 128):
            ix = val[lo:lo + 128]
            key = ids[ix]
            alternatives = np.sort(pools[key], axis=1)
            unique = np.ones_like(alternatives, dtype=bool)
            unique[:, 1:] = alternatives[:, 1:] != alternatives[:, :-1]
            candidates = np.concatenate([key[:, None], alternatives], axis=1)
            bx = features[ix].to(device).float()
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                spec, logits = model.encode_spectrum(bx)
                structures = model.encode_structure(fps[torch.as_tensor(candidates, device=device)])
            scores = torch.einsum('bd,bnd->bn', spec, structures)
            rank = 1 + ((scores[:, 1:] >= scores[:, :1])
                        & torch.as_tensor(unique, device=device)).sum(1)
            reciprocal += float((1.0 / rank.float()).sum())
            loss = F.binary_cross_entropy_with_logits(logits.float() - log_weight,
                                                      fps[torch.as_tensor(key, device=device)].float())
            bce += float(loss) * len(ix)
            count += len(ix)
    return bce / max(count, 1), reciprocal / max(count, 1)


def train_contrastive(base, log_weight, x, ids, weights, holdout, fps, masses, formulas,
                      seed, deadline):
    torch.manual_seed(seed + 101)
    torch.cuda.manual_seed_all(seed + 101)
    rng = np.random.default_rng(seed + 102)
    device = next(base.parameters()).device
    model = DualEncoder(base).to(device)
    # No holdout keys in supervised positives, hard pools, or in-batch negatives.
    training_keys = np.unique(ids[~holdout])
    pools = negative_table(masses, formulas, training_keys, seed + 103)
    # Validation has distinct mass-window candidates, including unseen holdout structures.
    val_pools = negative_table(masses, formulas, np.arange(len(fps)), seed + 104, width=128)
    val = np.flatnonzero(holdout)
    if len(val) > 2048:
        # Per-structure balanced, mixed-source holdout sampling.
        vw = weights[val].copy()
        vw /= vw.sum()
        val = rng.choice(val, 2048, replace=False, p=vw)
    features = torch.from_numpy(x)
    targets = torch.from_numpy(fps).to(device)
    probs = weights.copy()
    probs[holdout] = 0
    probs /= probs.sum()
    cdf = np.cumsum(probs)
    frequency = np.clip(fps[training_keys].mean(0), 0.001, 0.999)
    pw = torch.from_numpy(np.minimum(np.sqrt((1 - frequency) / frequency), 4).astype(np.float32)).to(device)
    optimizer = torch.optim.AdamW([
        {'params': model.spectrum.parameters(), 'lr': 0.0001},
        {'params': list(model.project.parameters()) + list(model.structure.parameters()), 'lr': 0.001}],
        weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    baseline_bce, _ = _evaluate(model, features, ids, targets, val, val_pools, log_weight, device)
    best, best_state, stale, seconds = -1, None, 0, 30
    for epoch in range(MAX_EPOCHS):
        if time.monotonic() + max(20, seconds * 1.15) > deadline:
            break
        start = time.monotonic()
        model.train()
        indices = np.searchsorted(cdf, rng.random(EPOCH_SAMPLES))
        losses = np.zeros(2)
        batches = 0
        for lo in range(0, len(indices), BATCH):
            if time.monotonic() > deadline - 8:
                break
            ix = indices[lo:lo + BATCH]
            key = ids[ix]
            negative_ids = pools[key[:, None], rng.integers(0, pools.shape[1], (len(ix), HARD_COUNT))]
            tx = torch.as_tensor(key, device=device)
            negatives = torch.as_tensor(negative_ids, device=device)
            bx = features[ix].to(device).float()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                spectrum, logits = model.encode_spectrum(bx)
                positive = model.encode_structure(targets[tx])
                hard = model.encode_structure(targets[negatives])
                bce = F.binary_cross_entropy_with_logits(logits, targets[tx].float(), pos_weight=pw)
                ranking = ranking_loss(spectrum, positive, hard, tx)
                loss = bce + RANK_WEIGHT * ranking
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses += [float(bce.detach()), float(ranking.detach())]
            batches += 1
        vbce, mrr = _evaluate(model, features, ids, targets, val, val_pools, log_weight, device)
        seconds = time.monotonic() - start
        safe = np.isfinite(vbce) and vbce <= baseline_bce * 1.10
        print(f'[contrast] epoch={epoch + 1}, bce={losses[0] / max(1, batches):.5f}, '
              f'rank_loss={losses[1] / max(1, batches):.5f}, holdout_bce={vbce:.5f}, '
              f'holdout_mass_mrr={mrr:.5f}, safe={safe}, seconds={seconds:.1f}', flush=True)
        if safe and mrr > best + 0.0005:
            best, best_state, stale = mrr, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if epoch in (7, 15, 23):
            for group in optimizer.param_groups:
                group['lr'] *= 0.5
        if stale >= 8 and epoch >= 11:
            break
    if best_state is None:
        raise RuntimeError('no safe contrastive checkpoint within budget')
    model.load_state_dict(best_state)
    print(f'[contrast] selected holdout mass MRR={best:.5f}, baseline BCE={baseline_bce:.5f}', flush=True)
    return model


def predict_embeddings(model, query_x, fps):
    device = next(model.parameters()).device
    model.eval()
    spec, structures = [], []
    with torch.no_grad():
        for lo in range(0, len(query_x), BATCH):
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                z, _ = model.encode_spectrum(torch.from_numpy(query_x[lo:lo + BATCH]).to(device).float())
            spec.append(z.cpu().numpy())
        for lo in range(0, len(fps), 4096):
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                z = model.encode_structure(torch.from_numpy(fps[lo:lo + 4096]).to(device))
            structures.append(z.cpu().numpy())
    return np.concatenate(spec), np.concatenate(structures)

In [ ]:
%%writefile src/features.py
"""Arrow/Numba spectral features: fragments, precursor-relative losses and metadata."""
from __future__ import annotations

import numpy as np
import pyarrow as pa
from numba import njit

from .masses import ADDUCTS, neutral_mass

BIN_WIDTH = 0.5
N_BINS = 2401  # 0..1200 Da
ADDUCT_NAMES = tuple(ADDUCTS)
N_META = 24
INPUT_DIM = 2 * N_BINS + N_META


@njit
def _bin_spectra(mz, it, offsets, selected, prec):
    out = np.zeros((len(selected), INPUT_DIM), dtype=np.float32)
    for j in range(len(selected)):
        i = selected[j]
        p = prec[i]
        maximum = 0.0
        for k in range(offsets[i], offsets[i + 1]):
            if np.isfinite(mz[k]) and np.isfinite(it[k]) and it[k] > maximum and mz[k] <= p + 2.0:
                maximum = it[k]
        if maximum <= 0:
            continue
        for k in range(offsets[i], offsets[i + 1]):
            if not np.isfinite(mz[k]) or not np.isfinite(it[k]) or mz[k] > p + 2.0:
                continue
            if it[k] < maximum * 0.005:
                continue
            w = np.sqrt(it[k] / maximum)
            b = int(np.rint(mz[k] / BIN_WIDTH))
            if 0 <= b < N_BINS:
                out[j, b] = max(out[j, b], w)
            b = int(np.rint((p - mz[k]) / BIN_WIDTH))
            if 0 <= b < N_BINS:
                out[j, N_BINS + b] = max(out[j, N_BINS + b], w)
        # L2 normalization per channel. Metadata carries scale and peak count.
        for start in (0, N_BINS):
            norm = np.sqrt(np.sum(out[j, start:start + N_BINS] ** 2))
            if norm > 0:
                out[j, start:start + N_BINS] /= norm
    return out


def feature_table(table, selected=None):
    """One contiguous Arrow row group; indices select training spectra before densification."""
    table = table.combine_chunks()
    if selected is None:
        selected = np.arange(len(table), dtype=np.int64)
    mz = table['ms2_mzs'].chunk(0)
    it = table['ms2_normalized_intensities'].chunk(0)
    prec = table['precursor_mz'].to_numpy(zero_copy_only=False).astype(np.float64)
    x = _bin_spectra(mz.values.to_numpy(), it.values.to_numpy(),
                     mz.offsets.to_numpy(), selected, prec)
    meta = x[:, 2 * N_BINS:]
    ads = table['adduct'].to_pylist()
    modes = table['ionization_mode'].to_pylist()
    energies = table['collision_energy_ev'].to_pylist()
    for j, i in enumerate(selected):
        ad = ads[i]
        ai = ADDUCT_NAMES.index(ad) if ad in ADDUCTS else len(ADDUCT_NAMES)
        meta[j, ai] = 1
        meta[j, 11] = modes[i] == 'positive'
        meta[j, 12] = modes[i] == 'negative'
        meta[j, 13] = np.clip(prec[i] / 1000, 0, 3) if np.isfinite(prec[i]) else 0
        mass = neutral_mass(prec[i], ad)
        meta[j, 14] = np.clip(mass / 1000, 0, 3) if np.isfinite(mass) else 0
        meta[j, 15] = not np.isfinite(mass)
        ce = energies[i]
        ce = np.asarray([] if ce is None else ce, dtype=float)
        ce = ce[np.isfinite(ce)]
        if len(ce):
            meta[j, 16:19] = np.clip([ce.mean() / 100, ce.min() / 100, ce.max() / 100], 0, 5)
            meta[j, 19] = min(len(ce), 5) / 5
        else:
            meta[j, 20] = 1
        meta[j, 21] = np.log1p(mz.offsets[i + 1].as_py() - mz.offsets[i].as_py()) / 8
        meta[j, 22] = not np.isfinite(prec[i])
        meta[j, 23] = modes[i] not in ('positive', 'negative')
    return x.astype(np.float16)


def query_features(q):
    cols = ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz', 'adduct',
            'ionization_mode', 'collision_energy_ev']
    return feature_table(pa.Table.from_pandas(q[cols], preserve_index=False))

In [ ]:
%%writefile src/fingerprints.py
"""Stereo-independent Morgan targets, rebuilt from permitted candidate SMILES."""
from __future__ import annotations

from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context

import numpy as np

FP_BITS = 2048


def _fingerprint_chunk(smiles):
    from rdkit import Chem, RDLogger
    from rdkit.Chem import rdFingerprintGenerator
    RDLogger.DisableLog('rdApp.*')
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS, includeChirality=False)
    out = np.zeros((len(smiles), FP_BITS), dtype=np.uint8)
    for i, smi in enumerate(smiles):
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            out[i] = gen.GetFingerprintAsNumPy(mol)
    return out


def build_fingerprints(smiles, workers=16):
    chunks = [smiles[i:i + 2048] for i in range(0, len(smiles), 2048)]
    # Avoid forking after Arrow/Torch have started threads; children never inherit CUDA state.
    with ProcessPoolExecutor(max_workers=workers, mp_context=get_context('spawn')) as pool:
        return np.concatenate(list(pool.map(_fingerprint_chunk, chunks)))

In [ ]:
%%writefile src/fragment.py
"""Independent fragment-only diagnostic; never replaces inherited library scores."""
from __future__ import annotations

from dataclasses import dataclass
import hashlib
import re

import numpy as np


@dataclass(frozen=True)
class FragmentPolicy:
    enabled: bool = False
    cutoff: float = .30


DEFAULT_FRAGMENT = FragmentPolicy(True, .50)


def identity(mode, adduct):
    if not isinstance(mode, str) or not isinstance(adduct, str):
        return None
    mode = mode.strip().lower()
    mode = {'pos': 'positive', '+': 'positive', 'neg': 'negative', '-': 'negative'}.get(mode, mode)
    adduct = re.sub(r'\s+', '', adduct).upper().replace(']1+', ']+').replace(']1-', ']-')
    if mode not in ('positive', 'negative') or not re.fullmatch(r'\[M[^\]]+\][+-]', adduct):
        return None
    if (mode == 'positive') != adduct.endswith('+'):
        return None
    return mode, adduct


def fragments(row, reference=False):
    """Return diagnostic masses/weights or unavailable, with precursor fraction."""
    try:
        precursor = float(row['precursor_mz'])
        mz = np.asarray(row['ms2_mzs'], dtype=float)
        it = np.asarray(row['ms2_normalized_intensities'], dtype=float)
        if not np.isfinite(precursor) or precursor <= 0 or mz.ndim != 1 or mz.shape != it.shape:
            return None
        valid = np.isfinite(mz) & np.isfinite(it) & (it > 0) & (mz > 0)
        mz, it = mz[valid], it[valid]
        if not len(it):
            return None
        with np.errstate(over='ignore', invalid='ignore'):
            total = it.sum()
        if not np.isfinite(total) or total <= 0:
            return None
        outside = np.abs(mz - precursor) > 1.5
        fraction = float(it[~outside].sum() / total)
        mz, it = mz[outside], it[outside]
        if not len(it):
            return None
        # Merge exact duplicates, so duplicated physical peaks cannot inflate support.
        unique, inverse = np.unique(mz, return_inverse=True)
        it = np.bincount(inverse, weights=it)
        mz = unique
        keep = it >= .01 * it.max()
        mz, it = mz[keep], it[keep]
        order = np.lexsort((mz, -it))[:64]
        mz, it = mz[order], it[order]
        if len(mz) < 3:
            return None
        # Entire fragment mass list rounded to <=2 decimal places is not adequate
        # evidence for this 0.005-Da test; it must never be treated as contradiction.
        if reference and np.all(np.abs(mz * 100 - np.rint(mz * 100)) < 1e-6):
            return None
        order = np.argsort(mz, kind='stable')
        w = np.sqrt(it[order])
        norm = np.linalg.norm(w)
        if not np.isfinite(norm) or norm <= 0:
            return None
        return mz[order], w / norm, fraction
    except (TypeError, ValueError, KeyError, OverflowError):
        return None


def direct_match(query, reference):
    """Maximum-weight deterministic one-to-one direct assignment, bounded cosine."""
    from scipy.optimize import linear_sum_assignment
    qm, qw, _ = query
    rm, rw, _ = reference
    tolerance = np.maximum(.005, qm[:, None] * 20e-6)
    allowed = np.abs(qm[:, None] - rm[None, :]) <= tolerance
    products = qw[:, None] * rw[None, :] * allowed
    qi, ri = linear_sum_assignment(products, maximize=True)
    keep = products[qi, ri] > 0
    qi, ri = qi[keep], ri[keep]
    return {'cosine': float(np.clip(products[qi, ri].sum(), 0, 1)),
            'matched_count': int(len(qi)),
            'query_fraction': float(np.clip((qw[qi] ** 2).sum(), 0, 1)),
            'reference_fraction': float(np.clip((rw[ri] ** 2).sum(), 0, 1))}


class FragmentVerifier:
    """Per-invocation prepared-reference and pair cache. Unavailable is not zero."""
    def __init__(self, library, deadline, clock):
        self.library, self.deadline, self.clock = library, deadline, clock
        self.prepared, self.pairs = {}, {}

    def check_time(self):
        if self.clock() >= self.deadline:
            raise TimeoutError('fragment verification deadline')

    def molecule(self, queries, key):
        spectra = []
        for q in queries:
            self.check_time()
            ident = identity(q.get('ionization_mode'), q.get('adduct'))
            fragment = fragments(q)
            best, usable = None, 0
            if ident is not None and fragment is not None:
                query_hash = hashlib.sha256(fragment[0].tobytes() + fragment[1].tobytes()).hexdigest()
                for i, ref in enumerate(self.library.get(key, ())):
                    self.check_time()
                    if identity(ref.get('ionization_mode'), ref.get('adduct')) != ident:
                        continue
                    cache_key = (key, i)
                    if cache_key not in self.prepared:
                        self.prepared[cache_key] = fragments(ref, reference=True)
                    r = self.prepared[cache_key]
                    if r is None:
                        continue
                    usable += 1
                    pair_key = (query_hash, key, i)
                    if pair_key not in self.pairs:
                        self.pairs[pair_key] = direct_match(fragment, r)
                    result = self.pairs[pair_key]
                    if best is None or result['cosine'] > best['cosine']:
                        best = dict(result, reference_index=i)
            spectra.append({'best': best, 'usable_references': usable,
                            'precursor_fraction': None if fragment is None else fragment[2]})
        return spectra

In [ ]:
%%writefile src/fragment_pass.py
"""Deadline-bounded optional release pass over a complete n006 submission."""
from __future__ import annotations

import hashlib
import json
import time

import numpy as np
import pandas as pd

from .absolute_gate import view_support
from .fragment import DEFAULT_FRAGMENT, FragmentVerifier


def release_decision(record, policy):
    info = {'released': False, 'votes': 0, 'n_spectra': record['n_spectra'], 'views': {}}
    if not policy.enabled or not record.get('eligible', False):
        return info
    views = [record.get(x) for x in ('fp', 'contrast', 'peak')]
    if any(x is None for x in views):
        return info
    n = len(record['keys'])
    for name, x in zip(('fp', 'contrast', 'peak'), views):
        try:
            if np.shape(x) != (n,):
                return info
        except (TypeError, ValueError):
            return info
        r = view_support(x, record['library_top'], record['challenger'], .5)
        info['views'][name] = r
        if r is None:
            return info
    support = sum(r['supports'] for r in info['views'].values()) >= 2
    spectra = record.get('diagnostic', [])
    if len(spectra) != record['n_spectra']:
        return info
    votes = [s.get('best') is not None and np.isfinite(s['best']['cosine'])
             and s['best']['cosine'] < policy.cutoff for s in spectra]
    info.update(votes=int(sum(votes)), spectrum_votes=votes, learned_support=bool(support))
    info['released'] = bool(support and sum(votes) * 2 > record['n_spectra'])
    return info


def optional_fragment_pass(out, work, deadline, q, windows, library, records,
                           policy=DEFAULT_FRAGMENT):
    """Parent already written. Any failure/timeout leaves its bytes untouched."""
    if not policy.enabled or time.monotonic() >= deadline - 5:
        return
    started = time.monotonic()
    verifier = FragmentVerifier(library, min(deadline - 5, started + 90), time.monotonic)
    rows = []
    all_records = []
    for mid, group in q.groupby('molecule_id', sort=False):
        verifier.check_time()
        z = records[mid]
        gate = z['gate']
        top = gate.get('library_top')
        full = z['post_peak_order']
        parent_order = z['order']
        eligible = bool(top is not None and gate.get('regime') == 'absolute'
                        and not gate.get('absolute', {}).get('released', False)
                        and not gate.get('released', False) and full and full[0] != top)
        record = dict(z, library_top=top, challenger=full[0] if full else None,
                      n_spectra=len(group), eligible=eligible, diagnostic=[])
        if eligible:
            record['diagnostic'] = verifier.molecule(group.to_dict('records'), z['keys'][top])
        decision = release_decision(record, policy)
        record['fragment'] = decision
        record['cutoff'] = policy.cutoff
        order = full[:25] if decision['released'] else parent_order
        guesses = windows[mid].iloc[order]['smiles'].tolist() or ['C']
        rows.append({'molecule_id': mid, 'smiles': ';'.join(guesses)})
        all_records.append(record)
    verifier.check_time()
    (work / 'fragment_pairs.json').write_text(json.dumps(
        {str(k): v for k, v in verifier.pairs.items()}))
    temporary = work / 'submission_fragment.csv'
    pd.DataFrame(rows).to_csv(temporary, index=False)
    verifier.check_time()
    temporary.replace(out)
    for record in all_records:
        print('[fragment] ' + json.dumps(record), flush=True)
    print('[fragment_sha256] ' + hashlib.sha256(out.read_bytes()).hexdigest(), flush=True)
    print(f'[fragment_timing] seconds={time.monotonic() - started:.3f}; '
          f'pairs={len(verifier.pairs)}; references={len(verifier.prepared)}', flush=True)

In [ ]:
%%writefile src/hard_negatives.py
"""Deterministic training-only mass/formula negative pools, with no holdout leakage."""
from __future__ import annotations

import numpy as np


def negative_table(masses, formulas, allowed, seed, width=64):
    """Half same-formula, half near-mass; widen to nearest masses if necessary.

    Rows outside allowed are evaluation-only. No row can contain itself, and
    no excluded key can occur in any pool. IDs are unique metric-key IDs.
    """
    rng = np.random.default_rng(seed)
    masses = np.asarray(masses, dtype=float)
    allowed = np.asarray(allowed, dtype=int)
    allowed = allowed[np.isfinite(masses[allowed])]
    if len(allowed) < 2:
        raise ValueError('too few training-only negative keys')
    order = allowed[np.argsort(masses[allowed], kind='stable')]
    sorted_mass = masses[order]
    groups = {}
    for i in allowed:
        if formulas[i]:
            groups.setdefault(str(formulas[i]), []).append(i)
    table = np.empty((len(masses), width), dtype=np.int32)
    exact_counts = []
    for i, mass in enumerate(masses):
        tolerance = max(abs(mass) * 1e-5, 0.003) if np.isfinite(mass) else 0
        center = np.searchsorted(sorted_mass, mass)
        lo, hi = np.searchsorted(sorted_mass, [mass - tolerance, mass + tolerance])
        near = order[lo:hi]
        near = near[near != i]
        exact_counts.append(len(near))
        if len(near) < 8:
            # At least the nearest 128 keys: mass-hard rather than arbitrary chemistry.
            lo, hi = max(0, center - 64), min(len(order), center + 64)
            near = order[lo:hi]
            near = near[near != i]
        same = np.asarray(groups.get(str(formulas[i]), []), dtype=np.int32)
        same = same[same != i]
        if not len(same):
            same = near
        if not len(near):
            near = allowed[allowed != i]
        table[i, :width // 2] = rng.choice(same, width // 2, replace=True)
        table[i, width // 2:] = rng.choice(near, width - width // 2, replace=True)
    print(f'[contrast] negative keys={len(allowed)}, median exact-window alternatives='
          f'{np.median(exact_counts):.0f}', flush=True)
    return table

In [ ]:
%%writefile src/learner.py
"""Budget-aware spectrum-to-Morgan MLP with key-disjoint training validation."""
from __future__ import annotations

import copy
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from torch import nn

from .features import INPUT_DIM, feature_table

MAX_PER_KEY = 6
SOURCE_WEIGHT = 2.0  # public spectra relative to Enveda, after per-key spectrum balancing
EPOCH_SAMPLES = 524288
BATCH_SIZE = 1024
MAX_EPOCHS = 80


def prepare_training(path, target_keys, work, seed, deadline, max_per_key=MAX_PER_KEY):
    rng = np.random.default_rng(seed)
    pf = pq.ParquetFile(path)
    cols = ['key14', 'ingest_lib', 'precursor_error_ppm']
    meta = pq.read_table(path, columns=cols).to_pandas()
    mapping = {k: i for i, k in enumerate(target_keys)}
    ids = meta['key14'].map(mapping).fillna(-1).to_numpy(dtype=np.int32)
    error = meta['precursor_error_ppm'].to_numpy(dtype=float)
    valid = (ids >= 0) & (~np.isfinite(error) | (np.abs(error) <= 50))
    indices = np.flatnonzero(valid)
    if max_per_key:
        order = np.lexsort((rng.random(len(indices)), ids[indices]))
        ordered = indices[order]
        groups = ids[ordered]
        starts = np.r_[True, groups[1:] != groups[:-1]]
        within = np.arange(len(groups)) - np.maximum.accumulate(np.where(starts, np.arange(len(groups)), 0))
        selected = np.sort(ordered[within < max_per_key])
    else:
        selected = indices
    # Optional exact-mass view reuses this selection and the identical holdout split.
    np.save(Path(work) / 'selected_rows.npy', selected)
    target_ids = ids[selected]
    counts = np.bincount(target_ids, minlength=len(target_keys))
    weights = 1.0 / counts[target_ids]
    public = meta['ingest_lib'].to_numpy()[selected] != 'enveda-180'
    weights *= np.where(public, SOURCE_WEIGHT, 1.0)
    holdout_keys = rng.random(len(target_keys)) < 0.02
    holdout = holdout_keys[target_ids]
    del meta, ids
    work = Path(work)
    x = np.memmap(work / 'features.f16', mode='w+', dtype=np.float16,
                  shape=(len(selected), INPUT_DIM))
    feature_cols = ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz',
                    'adduct', 'ionization_mode', 'collision_energy_ev']
    cursor = offset = 0
    for rg in range(pf.num_row_groups):
        if time.monotonic() > deadline:
            raise TimeoutError('feature deadline; baseline remains available')
        n = pf.metadata.row_group(rg).num_rows
        hi = np.searchsorted(selected, offset + n)
        local = selected[cursor:hi] - offset
        if len(local):
            table = pf.read_row_group(rg, columns=feature_cols)
            x[cursor:hi] = feature_table(table, local)
        cursor, offset = hi, offset + n
    x.flush()
    print(f'[learn] {len(selected)} spectra, {len(np.unique(target_ids))} keys, '
          f'public fraction={public.mean():.3f}, holdout={holdout.sum()}', flush=True)
    return x, target_ids, weights, holdout, public


class FingerprintMLP(nn.Module):
    def __init__(self, bias):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(INPUT_DIM, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(2048, 2048), nn.BatchNorm1d(2048), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(2048, 1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(1024, 2048))
        self.network[-1].bias.data.copy_(bias)

    def forward(self, x):
        return self.network(x)


def train_model(x, ids, weights, holdout, fps, seed, deadline):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(8)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    rng = np.random.default_rng(seed + 1)
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    training_keys = np.unique(ids[~holdout])
    freq = np.clip(fps[training_keys].mean(axis=0), 0.001, 0.999)
    pos_weight = np.minimum(np.sqrt((1 - freq) / freq), 4).astype(np.float32)
    bias = torch.from_numpy(np.log(freq / (1 - freq)) + np.log(pos_weight)).float()
    model = FingerprintMLP(bias).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    pw = torch.from_numpy(pos_weight).to(device)
    y = torch.from_numpy(fps).to(device)
    # A 3090 can keep the complete capped spectral corpus on-device. Smaller GPUs use CPU gathering.
    on_gpu = False
    if device.type == 'cuda':
        free, _ = torch.cuda.mem_get_info()
        on_gpu = x.nbytes < free * 0.70
    features = torch.from_numpy(x)
    if on_gpu:
        features = features.to(device)
    print(f'[learn] device={device}, GPU corpus={on_gpu}, bytes={x.nbytes}', flush=True)
    tids = torch.from_numpy(ids.astype(np.int64)).to(device)
    probs = weights.copy()
    probs[holdout] = 0
    probs /= probs.sum()
    cdf = np.cumsum(probs)
    val = np.flatnonzero(holdout)
    if len(val) > 8192:
        val = rng.choice(val, 8192, replace=False)
    best, stale, best_state = float('inf'), 0, None
    epoch_seconds = 0
    for epoch in range(MAX_EPOCHS):
        if time.monotonic() + max(15, epoch_seconds * 1.2) > deadline:
            break
        start = time.monotonic()
        model.train()
        indices = np.searchsorted(cdf, rng.random(EPOCH_SAMPLES))
        total = 0
        for lo in range(0, len(indices), BATCH_SIZE):
            if time.monotonic() > deadline:
                break
            ix = indices[lo:lo + BATCH_SIZE]
            tx = torch.as_tensor(ix, device=device)
            bx = features[tx if on_gpu else ix].to(device, non_blocking=True)
            target = y[tids[tx]].float()
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(bx.float())
                loss = nn.functional.binary_cross_entropy_with_logits(logits, target, pos_weight=pw)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total += float(loss.detach())
        model.eval()
        vloss = 0.0
        with torch.no_grad():
            for lo in range(0, len(val), BATCH_SIZE):
                ix = val[lo:lo + BATCH_SIZE]
                tx = torch.as_tensor(ix, device=device)
                bx = features[tx if on_gpu else ix].to(device)
                with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                    logit = model(bx.float()).float() - pw.log()
                loss = nn.functional.binary_cross_entropy_with_logits(logit, y[tids[tx]].float())
                vloss += float(loss) * len(ix)
        vloss /= max(len(val), 1)
        epoch_seconds = time.monotonic() - start
        print(f'[learn] epoch={epoch + 1}, train={total / max(1, len(indices) // BATCH_SIZE):.5f}, '
              f'holdout_bce={vloss:.5f}, seconds={epoch_seconds:.1f}', flush=True)
        if np.isfinite(vloss) and vloss < best - 0.00001:
            best, stale = vloss, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= 5 and epoch >= 7:
            break
        # Scheduled decay after the initial representation learning phase.
        if epoch in (9, 19, 39, 59):
            for group in optimizer.param_groups:
                group['lr'] *= 0.5
    if best_state is None:
        raise RuntimeError('no trained epoch before deadline')
    model.load_state_dict(best_state)
    del features, y, tids
    return model, pw.log()


def predict(model, log_weight, features):
    device = next(model.parameters()).device
    model.eval()
    predictions = []
    with torch.no_grad():
        for lo in range(0, len(features), BATCH_SIZE):
            x = torch.from_numpy(features[lo:lo + BATCH_SIZE]).to(device).float()
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                logits = model(x).float() - log_weight
            predictions.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(predictions)

In [ ]:
%%writefile src/library_gate.py
"""Observable two-view disagreement veto for weak library promotions only."""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np


@dataclass(frozen=True)
class VetoPolicy:
    enabled: bool = False
    margin: float = 1.0  # within-window SD units, using the inherited 0.025 floor
    rank_cutoff: int = 5  # library winner must be strictly below this rank in BOTH views
    agreement_fraction: float = 0.0  # optional fraction of spectra satisfying the SAME veto


DEFAULT_VETO = VetoPolicy(enabled=True, margin=0.5, rank_cutoff=3)


def disagreement(fp, contrast, library_top, blended_top, policy,
                 spectrum_fp=None, spectrum_contrast=None):
    """Release only when both views substantially favor the SAME blended winner.

    Strict-greater competition ranks ensure tied candidates are not counted as
    evidence against the library winner. Missing/invalid contrast keeps n002.
    """
    info = {'released': False, 'fp_rank': None, 'contrast_rank': None,
            'fp_margin': None, 'contrast_margin': None}
    if contrast is None or library_top == blended_top:
        return info
    fp, contrast = np.asarray(fp, dtype=float), np.asarray(contrast, dtype=float)
    if not np.isfinite(fp).all() or not np.isfinite(contrast).all():
        return info
    info.update(fp_rank=int(1 + (fp > fp[library_top]).sum()),
                contrast_rank=int(1 + (contrast > contrast[library_top]).sum()),
                fp_margin=float((fp[blended_top] - fp[library_top]) / max(fp.std(), 0.025)),
                contrast_margin=float((contrast[blended_top] - contrast[library_top]) /
                                      max(contrast.std(), 0.025)))
    info['released'] = bool(policy.enabled
                            and info['fp_rank'] > policy.rank_cutoff
                            and info['contrast_rank'] > policy.rank_cutoff
                            and info['fp_margin'] >= policy.margin
                            and info['contrast_margin'] >= policy.margin)
    if policy.agreement_fraction and info['released']:
        if spectrum_fp is None or spectrum_contrast is None or not len(spectrum_fp):
            info['released'] = False
        else:
            per_view = VetoPolicy(True, policy.margin, policy.rank_cutoff)
            votes = [disagreement(f, c, library_top, blended_top, per_view)['released']
                     for f, c in zip(spectrum_fp, spectrum_contrast)]
            info['spectrum_agreement'] = float(np.mean(votes))
            info['released'] = bool(info['spectrum_agreement'] >= policy.agreement_fraction)
    return info

In [ ]:
%%writefile src/masses.py
"""Adduct arithmetic: precursor m/z -> neutral monoisotopic mass of the molecule M."""
from __future__ import annotations

import numpy as np

PROTON = 1.007276
ELECTRON = 0.000549
H2O = 18.010565

# adduct -> (multiplier of M, mass added to n*M); all test adducts are singly charged
ADDUCTS = {
    '[M+H]+': (1, PROTON),
    '[M+NH4]+': (1, 18.033823),
    '[M-H2O+H]+': (1, PROTON - H2O),
    '[M-2H2O+H]+': (1, PROTON - 2 * H2O),
    '[M+Na]+': (1, 22.989218),
    '[M+K]+': (1, 38.963158),
    '[M-H]-': (1, -PROTON),
    '[M-H2O-H]-': (1, -PROTON - H2O),
    '[M+CH2O2-H]-': (1, 46.005479 - PROTON),
    '[M+Cl]-': (1, 34.968853 + ELECTRON),
}


def neutral_mass(precursor_mz: float, adduct: str) -> float:
    """NaN for adducts outside the table."""
    if adduct not in ADDUCTS:
        return float('nan')
    n, delta = ADDUCTS[adduct]
    return (float(precursor_mz) - delta) / n


def consensus_mass(masses) -> float:
    """Median over a molecule's spectra, ignoring unknown adducts."""
    arr = np.asarray([m for m in masses if np.isfinite(m)], dtype=float)
    return float(np.median(arr)) if arr.size else float('nan')

In [ ]:
%%writefile src/peak_config.py
"""Single shared optional-view blend coefficient, selected on public development."""
BLEND_WEIGHT = 1.0

In [ ]:
%%writefile src/peak_encoder.py
"""Small precision-preserving peak-set dual encoder, trained from scratch."""
from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

WIDTH = 96
EMBED_DIM = 256


class PeakEncoder(nn.Module):
    def __init__(self, bias, width=WIDTH):
        super().__init__()
        # Explicit integer/defect scales complement logarithmically spaced scales.
        frequencies = torch.cat([torch.logspace(-3, 2, 24), torch.tensor([1., 2., 4., 8.])])
        self.register_buffer('frequencies', frequencies)
        self.peak_project = nn.Linear(4 * len(frequencies) + 7, width)
        self.blocks = nn.ModuleList([nn.TransformerEncoderLayer(
            width, 4, width * 2, dropout=0.1, activation='gelu', batch_first=True,
            norm_first=True) for _ in range(2)])
        self.hidden = nn.Sequential(nn.Linear(width * 2 + 24, 512), nn.ReLU(), nn.Dropout(0.1),
                                    nn.Linear(512, 256), nn.ReLU())
        self.fingerprint = nn.Linear(256, 2048)
        self.fingerprint.bias.data.copy_(bias)
        self.project = nn.Linear(256, EMBED_DIM)
        self.structure = nn.Sequential(nn.Linear(2048, 512), nn.ReLU(), nn.Linear(512, EMBED_DIM))

    def mass_features(self, peaks):
        # Do not let autocast/half storage erase the sub-bin mass information.
        with torch.autocast(peaks.device.type, enabled=False):
            p = peaks.float()
            mass = p[..., :2]
            phase = mass[..., None] * self.frequencies.float() * (2 * torch.pi)
            fourier = torch.cat([phase.sin(), phase.cos()], dim=-1).flatten(-2)
            intensity = p[..., 2:3].clamp(0, 1)
            precursor = mass.sum(-1, keepdim=True)
            scalars = torch.cat([mass / 1000, mass / precursor.clamp_min(1),
                                 intensity, intensity.sqrt(), torch.log1p(intensity * 100) / 5], -1)
            return torch.cat([fourier, scalars], -1)

    def encode_spectrum(self, peaks, metadata):
        valid = peaks[..., 2] > 0
        # An empty spectrum has a single zero sentinel, avoiding all-masked attention NaNs.
        mask = valid.clone()
        mask[:, 0] |= ~valid.any(1)
        features = self.mass_features(peaks)
        features = features * valid[..., None]
        h = self.peak_project(features)
        for block in self.blocks:
            h = block(h, src_key_padding_mask=~mask)
        hf = h.float()
        mean = (hf * mask[..., None]).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        weights = peaks[..., 2].float().sqrt() * valid
        weights = weights / weights.sum(1, keepdim=True).clamp_min(1e-8)
        weighted = (hf * weights[..., None]).sum(1)
        hidden = self.hidden(torch.cat([mean, weighted, metadata.float()], -1))
        return F.normalize(self.project(hidden).float(), dim=-1), self.fingerprint(hidden)

    def encode_structure(self, fp):
        return F.normalize(self.structure(fp.float()).float(), dim=-1)

In [ ]:
%%writefile src/peak_features.py
"""Deterministic float32 exact-mass peak sets; padding has intensity zero."""
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from numba import njit

PEAK_COUNT = 64
COLUMNS = ['ms2_mzs', 'ms2_normalized_intensities', 'precursor_mz']


@njit
def select_peaks(mz, intensity, offsets, selected, precursor, count):
    out = np.zeros((len(selected), count, 3), dtype=np.float32)
    for j, i in enumerate(selected):
        p = precursor[i]
        if not np.isfinite(p) or p <= 0:
            continue
        lo, hi = offsets[i], offsets[i + 1]
        values = np.zeros(hi - lo, dtype=np.float64)
        for k in range(lo, hi):
            if (np.isfinite(mz[k]) and 0 < mz[k] <= p + 2 and
                    np.isfinite(intensity[k]) and intensity[k] > 0):
                values[k - lo] = intensity[k]
        maximum = np.max(values) if len(values) else 0.0
        if maximum <= 0:
            continue
        # Ties are resolved deterministically by the input's fixed peak order.
        order = np.argsort(-values)
        for t in range(min(count, len(order))):
            k = order[t]
            if values[k] < maximum * 0.005:
                break
            out[j, t, 0] = mz[lo + k]
            out[j, t, 1] = p - mz[lo + k]
            out[j, t, 2] = values[k] / maximum
    return out


def peak_table(table, selected=None, count=PEAK_COUNT):
    table = table.combine_chunks()
    if selected is None:
        selected = np.arange(len(table), dtype=np.int64)
    mz, it = table['ms2_mzs'].chunk(0), table['ms2_normalized_intensities'].chunk(0)
    if not np.array_equal(mz.offsets.to_numpy(), it.offsets.to_numpy()):
        raise ValueError('unaligned peak arrays')
    return select_peaks(mz.values.to_numpy(), it.values.to_numpy(), mz.offsets.to_numpy(),
                        selected, table['precursor_mz'].to_numpy().astype(np.float64), count)


def prepare_peaks(path, work, deadline, count=PEAK_COUNT):
    work = Path(work)
    selected = np.load(work / 'selected_rows.npy')
    peaks = np.memmap(work / 'peaks.f32', mode='w+', dtype=np.float32,
                      shape=(len(selected), count, 3))
    pf = pq.ParquetFile(path)
    cursor = offset = 0
    for rg in range(pf.num_row_groups):
        if time.monotonic() > deadline:
            raise TimeoutError('optional peak extraction deadline')
        n = pf.metadata.row_group(rg).num_rows
        hi = np.searchsorted(selected, offset + n)
        if hi > cursor:
            peaks[cursor:hi] = peak_table(pf.read_row_group(rg, columns=COLUMNS),
                                         selected[cursor:hi] - offset, count)
        cursor, offset = hi, offset + n
    peaks.flush()
    if time.monotonic() > deadline:
        raise TimeoutError('optional peak extraction completion deadline')
    print(f'[peak] extracted {len(peaks)} float32 peak sets', flush=True)
    return peaks


def query_peaks(q, count=PEAK_COUNT):
    return peak_table(pa.Table.from_pandas(q[COLUMNS], preserve_index=False), count=count)

In [ ]:
%%writefile src/peak_train.py
"""Budget-bounded joint fingerprint/contrastive training with key-disjoint holdout."""
from __future__ import annotations

import copy
import time

import numpy as np
import torch
from torch.nn import functional as F

from .contrastive import ranking_loss
from .hard_negatives import negative_table
from .peak_encoder import PeakEncoder, WIDTH

BATCH = 256
EPOCH_SAMPLES = 131072
MAX_EPOCHS = 80
RANK_WEIGHT = 0.20


def evaluate(model, peaks, meta, ids, targets, val, pools, log_weight, device, deadline):
    model.eval()
    reciprocal = bce = count = 0
    with torch.no_grad():
        for lo in range(0, len(val), 128):
            if time.monotonic() > deadline:
                raise TimeoutError('optional peak validation deadline')
            ix = val[lo:lo + 128]
            key = ids[ix]
            alternatives = np.sort(pools[key], axis=1)
            unique = np.ones_like(alternatives, dtype=bool)
            unique[:, 1:] = alternatives[:, 1:] != alternatives[:, :-1]
            candidates = np.concatenate([key[:, None], alternatives], axis=1)
            with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == 'cuda'):
                spec, logits = model.encode_spectrum(peaks[ix].to(device), meta[ix].to(device))
                structures = model.encode_structure(targets[torch.as_tensor(candidates, device=device)])
            scores = torch.einsum('bd,bnd->bn', spec, structures)
            rank = 1 + ((scores[:, 1:] >= scores[:, :1]) &
                        torch.as_tensor(unique, device=device)).sum(1)
            reciprocal += float((1 / rank.float()).sum())
            loss = F.binary_cross_entropy_with_logits(logits.float() - log_weight,
                                                      targets[torch.as_tensor(key, device=device)].float())
            bce += float(loss) * len(ix)
            count += len(ix)
    return bce / max(count, 1), reciprocal / max(count, 1)


def train_peak(peaks, metadata, ids, weights, holdout, fps, masses, formulas,
               seed, deadline, width=WIDTH, max_epochs=MAX_EPOCHS):
    if time.monotonic() > deadline - 25:
        raise TimeoutError('optional peak training deadline')
    torch.manual_seed(seed + 401)
    torch.cuda.manual_seed_all(seed + 401)
    torch.set_num_threads(8)
    if not torch.cuda.is_available():
        raise RuntimeError('optional attention encoder requires CUDA throughput')
    device = torch.device('cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')
    rng = np.random.default_rng(seed + 402)
    training_keys = np.unique(ids[~holdout])
    freq = np.clip(fps[training_keys].mean(0), 0.001, 0.999)
    pw = torch.from_numpy(np.minimum(np.sqrt((1 - freq) / freq), 4).astype(np.float32)).to(device)
    bias = torch.from_numpy(np.log(freq / (1 - freq)).astype(np.float32)).to(device) + pw.log()
    model = PeakEncoder(bias, width).to(device)
    pools = negative_table(masses, formulas, training_keys, seed + 403)
    val_pools = negative_table(masses, formulas, np.arange(len(fps)), seed + 404, width=128)
    if time.monotonic() > deadline - 25:
        raise TimeoutError('optional peak pool preparation deadline')
    val = np.flatnonzero(holdout)
    if len(val) > 2048:
        vw = weights[val].copy()
        vw /= vw.sum()
        val = rng.choice(val, 2048, replace=False, p=vw)
    features = torch.from_numpy(peaks)
    meta = torch.from_numpy(np.ascontiguousarray(metadata, dtype=np.float32))
    free, _ = torch.cuda.mem_get_info(device)
    on_gpu = features.nbytes + meta.nbytes < free * 0.50
    if on_gpu:
        features, meta = features.to(device), meta.to(device)
    targets = torch.from_numpy(fps).to(device)
    probs = weights.copy()
    probs[holdout] = 0
    probs /= probs.sum()
    cdf = np.cumsum(probs)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    best, state, stale, seconds = -1, None, 0, 40
    print(f'[peak] device={device}, width={width}, peaks={peaks.shape[1]}, GPU corpus={on_gpu}', flush=True)
    for epoch in range(max_epochs):
        if time.monotonic() + max(25, seconds * 1.15) > deadline:
            break
        start = time.monotonic()
        model.train()
        indices = np.searchsorted(cdf, rng.random(EPOCH_SAMPLES))
        losses = np.zeros(2)
        batches = 0
        for lo in range(0, len(indices), BATCH):
            if time.monotonic() > deadline - 15:
                break
            ix = indices[lo:lo + BATCH]
            key = ids[ix]
            negative_ids = pools[key[:, None], rng.integers(0, pools.shape[1], (len(ix), 16))]
            tx = torch.as_tensor(key, device=device)
            negatives = torch.as_tensor(negative_ids, device=device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16):
                spectrum, logits = model.encode_spectrum(features[ix].to(device), meta[ix].to(device))
                positive = model.encode_structure(targets[tx])
                hard = model.encode_structure(targets[negatives])
                bce = F.binary_cross_entropy_with_logits(logits, targets[tx].float(), pos_weight=pw)
                ranking = ranking_loss(spectrum, positive, hard, tx)
                loss = bce + RANK_WEIGHT * ranking
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            losses += [float(bce.detach()), float(ranking.detach())]
            batches += 1
        vbce, mrr = evaluate(model, features, meta, ids, targets, val, val_pools,
                             pw.log(), device, deadline)
        seconds = time.monotonic() - start
        print(f'[peak] epoch={epoch + 1}, bce={losses[0] / max(1, batches):.5f}, '
              f'rank_loss={losses[1] / max(1, batches):.5f}, holdout_bce={vbce:.5f}, '
              f'holdout_mass_mrr={mrr:.5f}, seconds={seconds:.1f}', flush=True)
        if np.isfinite(vbce) and np.isfinite(mrr) and mrr > best + 0.0005:
            best, state, stale = mrr, copy.deepcopy(model.state_dict()), 0
        else:
            stale += 1
        if epoch in (15, 31, 47, 63):
            for group in optimizer.param_groups:
                group['lr'] *= 0.5
        if stale >= 12 and epoch >= 23:
            break
    if state is None:
        raise RuntimeError('no finite peak checkpoint before optional deadline')
    model.load_state_dict(state)
    print(f'[peak] selected holdout mass MRR={best:.5f}', flush=True)
    return model


def predict_peak(model, peaks, metadata, fps, deadline):
    device = next(model.parameters()).device
    model.eval()
    spec, structures = [], []
    with torch.no_grad():
        for lo in range(0, len(peaks), BATCH):
            if time.monotonic() > deadline:
                raise TimeoutError('peak query inference deadline')
            with torch.autocast('cuda', dtype=torch.float16):
                z, _ = model.encode_spectrum(torch.from_numpy(peaks[lo:lo + BATCH]).to(device),
                                             torch.from_numpy(metadata[lo:lo + BATCH]).to(device))
            spec.append(z.cpu().numpy())
        for lo in range(0, len(fps), 4096):
            if time.monotonic() > deadline:
                raise TimeoutError('peak candidate inference deadline')
            with torch.autocast('cuda', dtype=torch.float16):
                z = model.encode_structure(torch.from_numpy(fps[lo:lo + 4096]).to(device))
            structures.append(z.cpu().numpy())
    spec, structures = np.concatenate(spec), np.concatenate(structures)
    if time.monotonic() > deadline:
        raise TimeoutError('optional peak inference completion deadline')
    if not np.isfinite(spec).all() or not np.isfinite(structures).all():
        raise ValueError('nonfinite optional embeddings')
    return spec, structures

In [ ]:
%%writefile src/rank.py
"""Baseline ranking: mass-window retrieval from the candidate DB, ordered by best library cosine, then a prior."""
from __future__ import annotations

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from .spectra import binned, clean, cosine

PPM = 10.0
MIN_TOL_DA = 0.003
MAX_GUESSES = 25
MAX_WINDOW = 3000          # candidates kept per molecule before scoring (closest in mass first)
NO_LIBRARY_SCORE = -1.0    # candidates without training spectra rank below any library match


class CandidateIndex:
    """Candidate structures sorted by neutral monoisotopic mass for window queries."""

    def __init__(self, cands: pd.DataFrame):
        self.df = cands.sort_values('mono_mass', kind='mergesort').reset_index(drop=True)
        self.mass = self.df['mono_mass'].to_numpy()

    def window(self, mass: float, ppm: float = PPM) -> pd.DataFrame:
        if not np.isfinite(mass):
            return self.df.iloc[0:0]
        tol = max(mass * ppm * 1e-6, MIN_TOL_DA)
        lo, hi = np.searchsorted(self.mass, [mass - tol, mass + tol])
        hits = self.df.iloc[lo:hi]
        if len(hits) > MAX_WINDOW:
            hits = hits.iloc[np.argsort(np.abs(hits['mono_mass'].to_numpy() - mass))[:MAX_WINDOW]]
        return hits


def load_library(train_path, keys: set[str], diagnostic_library=None) -> dict[str, list[tuple[str, dict]]]:
    """Training spectra of the given structures, keyed by key14 -> [(ionization_mode, binned spectrum)]."""
    if not keys:
        return {}
    cols = ['key14', 'ionization_mode', 'precursor_mz', 'ms2_mzs', 'ms2_normalized_intensities']
    extra = ['adduct'] if diagnostic_library is not None else []
    table = pq.read_table(train_path, columns=cols + extra, filters=[('key14', 'in', sorted(keys))])
    lib: dict[str, list[tuple[str, dict]]] = {}
    adducts = table.column('adduct').to_pylist() if extra else [None] * len(table)
    for (key, mode, prec, mzs, its), adduct in zip(
            zip(*(table.column(c).to_pylist() for c in cols)), adducts):
        if diagnostic_library is not None:
            diagnostic_library.setdefault(key, []).append(dict(
                ionization_mode=mode, adduct=adduct, precursor_mz=prec,
                ms2_mzs=mzs, ms2_normalized_intensities=its))
        mz, w = clean(mzs, its, prec if prec is not None else np.inf)
        if w.size:
            lib.setdefault(key, []).append((mode, binned(mz, w)))
    return lib


def rank_molecule(queries: list[tuple[str, dict]], hits: pd.DataFrame, lib: dict) -> list[str]:
    """queries: [(ionization_mode, binned spectrum)] of one molecule. Returns up to 25 SMILES, best first."""
    scored = []
    for key, smiles, prior in zip(hits['key14'], hits['smiles'], hits['np_likeness']):
        best = NO_LIBRARY_SCORE
        for mode, ref in lib.get(key, ()):
            for qmode, q in queries:
                if qmode == mode:
                    best = max(best, cosine(q, ref))
        tie = float(prior) if prior is not None and np.isfinite(prior) else -10.0
        scored.append((best, tie, key, smiles))
    scored.sort(key=lambda t: (t[0], t[1]), reverse=True)
    out, seen = [], set()
    for _, _, key, smiles in scored:
        if key not in seen:
            seen.add(key)
            out.append(smiles)
        if len(out) == MAX_GUESSES:
            break
    return out

In [ ]:
%%writefile src/spectra.py
"""Spectrum cleaning and a binned cosine similarity (sqrt intensities, 0.01 Da bins)."""
from __future__ import annotations

import numpy as np

BIN_DA = 0.01
MIN_REL_INTENSITY = 0.01
TOP_N = 64


def clean(mzs, intensities, precursor_mz: float) -> tuple[np.ndarray, np.ndarray]:
    """Drop peaks above precursor + 2 Da and below 1% of the base peak; keep the top-N; L2-normalised sqrt weights."""
    mz = np.asarray(mzs, dtype=np.float64)
    it = np.asarray(intensities, dtype=np.float64)
    keep = (mz <= precursor_mz + 2.0) & np.isfinite(mz) & np.isfinite(it) & (it > 0)
    mz, it = mz[keep], it[keep]
    if it.size == 0:
        return mz, it
    keep = it >= MIN_REL_INTENSITY * it.max()
    mz, it = mz[keep], it[keep]
    if it.size > TOP_N:
        idx = np.argsort(it)[-TOP_N:]
        mz, it = mz[idx], it[idx]
    w = np.sqrt(it)
    return mz, w / np.linalg.norm(w)


def binned(mz: np.ndarray, w: np.ndarray) -> dict[int, float]:
    out: dict[int, float] = {}
    for b, x in zip(np.rint(mz / BIN_DA).astype(np.int64).tolist(), w.tolist()):
        out[b] = out.get(b, 0.0) + x
    return out


def cosine(a: dict[int, float], b: dict[int, float]) -> float:
    """Neighbouring bins also match, tolerating ~0.01 Da calibration differences between instruments."""
    if len(a) > len(b):
        a, b = b, a
    total = 0.0
    for k, x in a.items():
        y = b.get(k)
        if y is None:
            y = max(b.get(k - 1, 0.0), b.get(k + 1, 0.0))
        total += x * y
    return total

## Build the data directory
All training spectra keyed by the metric key, plus the candidate structure DB (training structures ∪ COCONUT).

In [ ]:
r = subprocess.run([sys.executable, 'kaggle_prepare.py', '--train', f'{COMP}/train.parquet', '--coconut', COCONUT,
                    '--out', f'{WORK}/data', '--procs', str(os.cpu_count())], capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, 'data preparation failed'
print(f'prepared in {time.time() - T0:.0f}s')

## Run the pipeline

In [ ]:
# Hard cap for the whole notebook is 9 h; the pipeline finishes by ~0.89 * BUDGET and writes a valid
# library-ranking submission before any learned stage, replacing it atomically when later stages succeed.
BUDGET = 25200
# The pipeline replaces its output atomically from the work dir, so both must be on one filesystem
# (/kaggle/working is a separate 20 GB volume); the result is copied there at the end.
import shutil
os.makedirs(f'{WORK}/work', exist_ok=True)
OUT = f'{WORK}/submission.csv'
cmd = [sys.executable, 'main.py', '--data-dir', f'{WORK}/data', '--queries', f'{COMP}/test.parquet',
       '--out', OUT, '--work-dir', f'{WORK}/work', '--budget', str(BUDGET), '--seed', '20261007']
with open('/kaggle/working/pipeline.log', 'w') as log:
    try:
        code = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, timeout=BUDGET + 600).returncode
    except subprocess.TimeoutExpired:
        code = 'timeout'
if os.path.exists(OUT):
    shutil.copy(OUT, '/kaggle/working/submission.csv')
print('pipeline exit', code, f'after {time.time() - T0:.0f}s')
print('stages:', [l.strip()[:90] for l in open('/kaggle/working/pipeline.log')
                  if l.startswith(('[blend]', '[peak] optional', '[fragment_timing]', 'Traceback'))])
print(''.join(l for l in open('/kaggle/working/pipeline.log') if l.startswith('[') and 'epoch=' not in l)[-6000:])

## Validate the submission

In [ ]:
import pandas as pd
test = pd.read_parquet(f'{COMP}/test.parquet', columns=['molecule_id'])
sub = pd.read_csv('/kaggle/working/submission.csv', dtype=str, keep_default_na=False)
assert list(sub.columns) == ['molecule_id', 'smiles'] and sub.molecule_id.is_unique
assert set(sub.molecule_id) == set(test.molecule_id), 'every test molecule needs exactly one row'
n = sub.smiles.str.split(';').map(lambda g: len([x for x in g if x.strip()]))
assert n.between(1, 25).all(), 'each row needs 1..25 guesses'
print(len(sub), 'molecules, mean guesses', round(n.mean(), 1), f'| total {time.time() - T0:.0f}s')
sub.head()